<a href="https://colab.research.google.com/github/diloo11/MachineLearning/blob/main/training_sepatu_condition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## IMPORT DRIVE

In [ ]:
from google.colab import drive
import os

# Hubungkan Google Drive
drive.mount('/content/drive')

Mounted at /content/drive


## BUAT FOLDER LANGSUNG

In [ ]:
from google.colab import drive
import os

# Hubungkan Google Drive
drive.mount('/content/drive')

# Buat folder khusus penyimpanan hasil training di Drive

drive_save_dir = '/content/drive/MyDrive/YOLO_Training_Shoe_Condition'
os.makedirs(drive_save_dir, exist_ok=True)

!pip install ultralytics roboflow



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 47.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.4/5.4 MB 88.2 MB/s  0:00:00
  Attempting uninstall: typer
    Found existing installation: typer 0.27.3
    Uninstalling typer-0.27.3:
      Successfully uninstalled typer-0.27.3
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [ultralytics]


## DOWNLOAD LIBRARY

In [ ]:
#!pip install ultralytics roboflow
#from ultralytics import YOLO
import torch
import os
from IPython.display import Image, clear_output

## IMPORT DATASET

In [ ]:
!pip install roboflow

from roboflow import Roboflow
rf = Roboflow(api_key="AxwVQz6sX7UHHvs0Qg0D")
project = rf.workspace("ichaa").project("shoe_condition1")
version = project.version(5)
dataset = version.download("yolov8")


loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to shoe_condition1-5 in yolov8:: 100%|██████████| 673/673 [00:00<00:00, 9418.29it/s]

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


## SPLITTING DATASET

In [ ]:
import os
import glob
import shutil
import yaml
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split

# ==============================================================================
# 1. KONFIGURASI PATH & RASIO
# ==============================================================================
# Sesuaikan path folder hasil extract dari Roboflow
SOURCE_DIR = "/content/shoe_condition1-5"
# Folder tujuan hasil split bersih (dibedakan agar tidak menimpa dataset material)
OUTPUT_DIR = "/content/dataset_split_clean_condition"

TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
TEST_RATIO = 0.15
RANDOM_SEED = 42  # Kunci seed agar pembagian selalu konsisten

# ==============================================================================
# 2. KUMPULKAN SEMUA GAMBAR & LABEL DARI TRAIN/VALID/TEST BAWAAN ROBOFLOW
# ==============================================================================
all_pairs = []  # Menyimpan tuple (path_gambar, path_label)

for split_folder in ["train", "valid", "test"]:
    img_dir = os.path.join(SOURCE_DIR, split_folder, "images")
    lbl_dir = os.path.join(SOURCE_DIR, split_folder, "labels")

    if not os.path.exists(img_dir):
        continue

    for img_path in glob.glob(os.path.join(img_dir, "*.*")):
        base_name = os.path.splitext(os.path.basename(img_path))[0]
        lbl_path = os.path.join(lbl_dir, base_name + ".txt")

        if os.path.exists(lbl_path):
            all_pairs.append((img_path, lbl_path))

print(f"Total pasangan gambar & label ditemukan: {len(all_pairs)} foto")

# ==============================================================================
# 3. TENTUKAN STRATIFICATION LABEL PER GAMBAR (AMAN DARI KELAS < 4 FOTO)
# ==============================================================================
# Hitung frekuensi global tiap kelas untuk menentukan kelas paling langka di tiap foto
global_class_counts = Counter()
image_classes = []

for _, lbl_path in all_pairs:
    classes_in_img = set()
    with open(lbl_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if parts:
                cls_id = int(parts[0])
                classes_in_img.add(cls_id)
                global_class_counts[cls_id] += 1
    image_classes.append(list(classes_in_img))

# Pilih kelas paling jarang (minoritas) sebagai kunci stratify
# Jika gambar kosong (background murni), beri label -1
raw_stratify_labels = []
for cls_list in image_classes:
    if not cls_list:
        raw_stratify_labels.append(-1)
    else:
        rarest_cls = min(cls_list, key=lambda c: global_class_counts[c])
        raw_stratify_labels.append(rarest_cls)

# Amankan kelas/grup yang jumlah fotonya < 4 agar tidak error saat dibagi ke Train/Val/Test
label_freq = Counter(raw_stratify_labels)
stratify_labels = [
    lbl if label_freq[lbl] >= 4 else -99
    for lbl in raw_stratify_labels
]

# ==============================================================================
# 4. EKSEKUSI STRATIFIED SPLIT (70% TRAIN : 15% VAL : 15% TEST)
# ==============================================================================
final_freq = Counter(stratify_labels)
use_stratify_1 = stratify_labels if min(final_freq.values()) >= 4 else None

# Tahap 1: Pisahkan 70% Train dan 30% Sisa (Val + Test)
train_pairs, temp_pairs, _, temp_labels = train_test_split(
    all_pairs,
    stratify_labels,
    test_size=(1.0 - TRAIN_RATIO),
    random_state=RANDOM_SEED,
    stratify=use_stratify_1
)

# Tahap 2: Bagi 30% Sisa menjadi 15% Val dan 15% Test (proporsi 0.5 : 0.5)
temp_freq = Counter(temp_labels)
use_stratify_2 = temp_labels if min(temp_freq.values()) >= 2 else None

val_relative_ratio = VAL_RATIO / (VAL_RATIO + TEST_RATIO)
val_pairs, test_pairs = train_test_split(
    temp_pairs,
    test_size=(1.0 - val_relative_ratio),
    random_state=RANDOM_SEED,
    stratify=use_stratify_2
)

splits = {
    "train": train_pairs,
    "valid": val_pairs,
    "test": test_pairs
}

# ==============================================================================
# 5. SALIN FILE KE FOLDER BARU & BUAT DATA.YAML
# ==============================================================================
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

for split_name, pairs in splits.items():
    os.makedirs(os.path.join(OUTPUT_DIR, split_name, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUTPUT_DIR, split_name, "labels"), exist_ok=True)

    for img_path, lbl_path in pairs:
        shutil.copy2(img_path, os.path.join(OUTPUT_DIR, split_name, "images", os.path.basename(img_path)))
        shutil.copy2(lbl_path, os.path.join(OUTPUT_DIR, split_name, "labels", os.path.basename(lbl_path)))

# Salin informasi kelas dari data.yaml lama ke data.yaml baru
old_yaml_path = os.path.join(SOURCE_DIR, "data.yaml")
with open(old_yaml_path, "r") as f:
    data_cfg = yaml.safe_load(f)

class_names = data_cfg["names"]
new_yaml = {
    "path": OUTPUT_DIR,
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": len(class_names),
    "names": class_names
}

new_yaml_path = os.path.join(OUTPUT_DIR, "data.yaml")
with open(new_yaml_path, "w") as f:
    yaml.dump(new_yaml, f, sort_keys=False)

# ==============================================================================
# 6. TAMPILKAN LAPORAN DISTRIBUSI INSTANSI PER KELAS
# ==============================================================================
print("\n" + "="*65)
print(f"{'Split':<10} | {'Foto':<6} | " + " | ".join([f"{name:<8}" for name in class_names]))
print("="*65)

for split_name, pairs in splits.items():
    cls_counter = Counter()
    for _, lbl_path in pairs:
        with open(lbl_path, "r") as f:
            for line in f:
                parts = line.strip().split()
                if parts:
                    cls_counter[int(parts[0])] += 1

    counts_str = " | ".join([f"{cls_counter[i]:<8}" for i in range(len(class_names))])
    print(f"{split_name:<10} | {len(pairs):<6} | {counts_str}")
print("="*65)
print(f"Dataset siap digunakan! File YAML tersimpan di: {new_yaml_path}")

Total pasangan gambar & label ditemukan: 334 foto

Split      | Foto   | color_change | detached | dirty    | normal   | rip      | yellowing
train      | 233    | 52       | 68       | 470      | 154      | 100      | 155     
valid      | 50     | 1        | 22       | 35       | 19       | 27       | 34      
test       | 51     | 4        | 19       | 30       | 25       | 24       | 29      
Dataset siap digunakan! File YAML tersimpan di: /content/dataset_split_clean_condition/data.yaml


## AUGMENTASI BLUR

In [ ]:
import os
import glob
import cv2
import shutil
import albumentations as A
from tqdm import tqdm

# ==============================================================================
# 1. KONFIGURASI FOLDER TRAIN
# ==============================================================================
TRAIN_IMG_DIR = "/content/dataset_split_clean_condition/train/images"
TRAIN_LBL_DIR = "/content/dataset_split_clean/train_condition/labels"

# Jumlah salinan augmentasi per 1 gambar asli di folder train
# Contoh: 2 berarti tiap 1 foto train asli akan menghasilkan 2 versi tambahan (total jadi 3x lipat)
AUG_MULTIPLIER = 2

# ==============================================================================
# 2. DEFINISI PIPELINE ALBUMENTATIONS (BLUR, NOISE, & PENCAHAYAAN RINGAN)
# ==============================================================================
# Transformasi ini murni mengubah tekstur piksel tanpa mengubah posisi poligon
transform = A.Compose([
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=(3, 5), p=1.0),
    ], p=0.6),  # 60% peluang terkena efek blur ringan (simulasi kamera goyang/kurang fokus)

    A.OneOf([
        A.GaussNoise(std_range=(0.02, 0.08), p=1.0),
        A.ISONoise(color_shift=(0.01, 0.03), intensity=(0.1, 0.3), p=1.0),
    ], p=0.6),  # 60% peluang terkena efek grain/noise sensor kamera

    # Variasi pencahayaan & kontras moderat (aman untuk tekstur material)
    A.RandomBrightnessContrast(brightness_limit=0.12, contrast_limit=0.12, p=0.5),
])

# ==============================================================================
# 3. EKSEKUSI AUGMENTASI HANYA PADA FOLDER TRAIN
# ==============================================================================
# Ambil daftar gambar asli sebelum ditambah file hasil augmentasi
original_images = [
    p for p in glob.glob(os.path.join(TRAIN_IMG_DIR, "*.*"))
    if "_aug" not in os.path.basename(p)
]

print(f"Jumlah gambar asli di folder train: {len(original_images)}")
created_count = 0

for img_path in tqdm(original_images, desc="Membuat Augmentasi Train"):
    base_name = os.path.splitext(os.path.basename(img_path))[0]
    ext = os.path.splitext(img_path)[1]
    lbl_path = os.path.join(TRAIN_LBL_DIR, base_name + ".txt")

    if not os.path.exists(lbl_path):
        continue

    # Baca gambar menggunakan OpenCV (konversi BGR ke RGB untuk Albumentations)
    image = cv2.imread(img_path)
    if image is None:
        continue
    image_rgb = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

    for i in range(1, AUG_MULTIPLIER + 1):
        # Terapkan augmentasi piksel
        augmented = transform(image=image_rgb)
        aug_image_bgr = cv2.cvtColor(augmented["image"], cv2.COLOR_RGB2BGR)

        # Simpan gambar baru & salin file label poligon .txt pasangannya
        new_base_name = f"{base_name}_aug{i}"
        new_img_path = os.path.join(TRAIN_IMG_DIR, new_base_name + ext)
        new_lbl_path = os.path.join(TRAIN_LBL_DIR, new_base_name + ".txt")

        cv2.imwrite(new_img_path, aug_image_bgr)
        shutil.copy2(lbl_path, new_lbl_path)
        created_count += 1

total_final_train = len(glob.glob(os.path.join(TRAIN_IMG_DIR, "*.*")))
print(f"\nSelesai! Berhasil membuat {created_count} gambar augmentasi baru.")
print(f"Total gambar di folder train sekarang : {total_final_train} foto")
print(f"Folder valid & test tetap 100% murni tanpa augmentasi.")

Jumlah gambar asli di folder train: 233


Membuat Augmentasi Train: 100%|██████████| 233/233 [00:00<00:00, 84146.10it/s]


Selesai! Berhasil membuat 0 gambar augmentasi baru.
Total gambar di folder train sekarang : 233 foto
Folder valid & test tetap 100% murni tanpa augmentasi.


## YOLOV8

In [ ]:
from ultralytics import YOLO


model = YOLO('yolov8n-seg.pt')

model.train(
    data=f"/content/shoe_condition1-4/data.yaml",
    epochs=50,
    patience=30,
    imgsz=640,
    project=drive_save_dir,  # Langsung simpan di Google Drive
    name='experiment_YOLOV8',     # Sub-folder eksperimen
    save=True,               # Wajib true untuk simpan checkpoint
    save_period=25            # Opsional: backup berkala tiap 5 epoch
)

Ultralytics 8.4.208 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/shoe_condition1-4/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=experiment-4, nbs=64, nms=None, opset=None, opti

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x77ff1dee6c80>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,  

## YOLOV11

In [ ]:
from ultralytics import YOLO


model = YOLO('yolo11n-seg.pt')

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    patience=30,
    imgsz=640,
    project=drive_save_dir,  # Langsung simpan di Google Drive
    name='experiment_YOLOV11',     # Sub-folder eksperimen
    save=True,               # Wajib true untuk simpan checkpoint
    save_period=25            # Opsional: backup berkala tiap 5 epoch
)

##YOLOV5

In [ ]:
from ultralytics import YOLO


model = YOLO('yolov5n-seg.pt')

model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    patience=30,
    imgsz=640,
    project=drive_save_dir,  # Langsung simpan di Google Drive
    name='experiment_YOLOV5',     # Sub-folder eksperimen
    save=True,               # Wajib true untuk simpan checkpoint
    save_period=25            # Opsional: backup berkala tiap 5 epoch
)

##YOLO26

In [ ]:
from ultralytics import YOLO


model = YOLO('yolo26n-seg.pt')

model.train(
    data=f"/content/dataset_split_clean_condition/data.yaml",
    epochs=50,
    patience=30,
    imgsz=640,
    dropout=0.1,          # Mematikan 10% neuron secara acak saat train agar tidak menghafal gambar
    weight_decay=0.001,   # Memberi penalti pada bobot yang terlalu besar (default biasanya 0.0005)
    shear=2.0,
    degrees=10.0,
    fliplr=0.5,
    hsv_v=0.2,      # Variasi pencahayaan/brightness ringan
    mosaic=1.0,
    project=drive_save_dir,  # Langsung simpan di Google Drive
    name='experiment_YOLO26',     # Sub-folder eksperimen
    save=True,               # Wajib true untuk simpan checkpoint
    save_period=25            # Opsional: backup berkala tiap 5 epoch
)

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset_split_clean_condition/data.yaml, degrees=10.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.1, dynamic=False, embed=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.2, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo26n-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=experiment_YOLO26, nbs=64, nms=None

ultralytics.utils.metrics.SegmentMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3, 4, 5])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7b21f17201a0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)', 'Precision-Recall(M)', 'F1-Confidence(M)', 'Precision-Confidence(M)', 'Recall-Confidence(M)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004

## BUAT DAPET AKURASI, RECALL, F1 SCORE

In [ ]:
# Muat model terbaik
model = YOLO('/content/drive/MyDrive/YOLO_Training_Shoe_Condition/experiment_YOLO26/weights/best.pt')

# Jalankan validasi pada test set
metrics = model.val(data=f"/content/dataset_split_clean_condition/data.yaml", split='test')

# Ambil rata-rata Precision dan Recall
precision = metrics.seg.mp
recall = metrics.seg.mr

# Hitung F1-Score secara matematis
f1_score = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0

print(f"Presisi (Mean Precision) : {precision:.4f}")
print(f"Recall (Mean Recall)       : {recall:.4f}")
print(f"F1-Score                   : {f1_score:.4f}")
print(f"mAP@50                     : {metrics.seg.map50:.4f}")

Ultralytics 8.4.216 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
YOLO26n-seg summary (fused): 137 layers, 2,704,998 parameters, 0 gradients, 9.2 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 864.1±308.3 MB/s, size: 30.5 KB)
val: Scanning /content/dataset_split_clean_condition/test/labels... 51 images, 0 backgrounds, 8 corrupt: 100% ━━━━━━━━━━━━ 51/51 1.4Kit/s 0.0s
val: /content/dataset_split_clean_condition/test/images/WhatsApp-Image-2026-11-09-at-20-11-44_jpeg.rf.fc4d8a8759abded0ef5d0ab8d68d036f.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/dataset_split_clean_condition/test/images/cracks_005_png_jpg.rf.9026ded753f178b1ea189003efefd7e4.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/dataset_split_clean_condition/test/images/cracks_032_png_jpg.rf.9bebced4aaa411bf9aeb8700440529fc.jpg: ignoring corrupt image/label: labels mix segment and detection rows
val: /content/dataset_split

In [ ]:
metrics = model.val()

print(metrics)

In [ ]:
from ultralytics import YOLO

# Ambil bobot terakhir yang sempat tersimpan di Drive
checkpoint_path = '/content/drive/MyDrive/YOLO_Training_Shoe/experiment_1/weights/last.pt'

model = YOLO(checkpoint_path)
model.train(resume=True)

In [ ]:
from ultralytics import YOLO


model_path = '/content/drive/MyDrive/YOLO_Training_Shoe/experiment_1/weights/best.pt'
model = YOLO(model_path)

# 2. Export ke format ONNX
model.export(
    format='onnx',
    imgsz=640,
    dynamic=False,
    simplify=True
)